# 00｜Colab 環境設定範本

**115-1 地理資訊系統運用程式**。這份 notebook 有兩個用途：

1. **第一次使用：** 確認 Google Drive 課程資料夾、資料讀取與中文字型都設定正確。
2. **之後每一份新 notebook：** 把「1. 路徑設定」與「3. 中文字型」兩個儲存格複製到最前面。

> 使用前請確認：課程資料夾 `115-1_GISprog` 已放在「我的雲端硬碟」最上層。

## 1. 路徑設定

- **在 Colab 執行：** 掛載 Google Drive，專案資料夾是 `/content/drive/MyDrive/115-1_GISprog`。
- **備援（在本機 VS Code 執行）：** 以 notebook 所在資料夾的上一層作為專案資料夾。

之後所有讀寫檔案都用 `RAW_DIR`、`PROCESSED_DIR`、`OUTPUT_DIR` 組合路徑，**不要寫死路徑**。

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/115-1_GISprog")
else:
    # 備援：本機執行時，notebook 位於 115-1_GISprog/notebooks/，上一層就是專案資料夾
    PROJECT_DIR = Path.cwd().parent

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"
FONT_DIR = PROJECT_DIR / "fonts"

if not RAW_DIR.exists():
    raise FileNotFoundError(f"找不到 {RAW_DIR}，請確認 115-1_GISprog 資料夾位置是否正確")

print("執行環境：", "Google Colab" if IN_COLAB else "本機")
print("專案資料夾：", PROJECT_DIR)
print("原始資料：", sorted(p.name for p in RAW_DIR.iterdir()))

## 2. 安裝額外套件（需要時才執行）

Colab 已內建 pandas、numpy、matplotlib、geopandas 等常用套件。課程後續需要其他套件時，再用 `%pip install` 安裝，例如 W5 的 leafmap。

- **重新連線後要重裝：** Colab 斷線或重新連線後，安裝的套件會消失，需要重新執行這個儲存格。
- **套件放在最前面：** 安裝指令請放在 notebook 最前面，方便別人重現你的結果。

In [ ]:
# 本週（W1）不需要額外套件；需要時把下一行的 # 拿掉並改成要安裝的套件
# %pip install -q leafmap

## 3. 中文字型（讓 matplotlib 圖表顯示中文）

- **只下載一次：** 字型存放在 Drive 的 `115-1_GISprog/fonts/`。第一次執行時才下載（約 20 MB），之後直接讀取。
- **沒有設定字型時：** 圖表中的中文會變成方框。

In [ ]:
import urllib.request

import matplotlib.pyplot as plt
from matplotlib import font_manager

FONT_URL = "https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_&export=download"
FONT_PATH = FONT_DIR / "TaipeiSansTCBeta-Regular.ttf"

if not FONT_PATH.exists():
    FONT_DIR.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(FONT_URL, FONT_PATH)

font_manager.fontManager.addfont(str(FONT_PATH))
plt.rcParams["font.family"] = "Taipei Sans TC Beta"
plt.rcParams["axes.unicode_minus"] = False  # 讓負號正常顯示
print("已設定中文字型：", FONT_PATH.name)

## 4. 讀取資料測試

讀取全球城市資料 `worldcities.csv`，確認路徑正確。資料來源見 `data/raw/資料來源說明.md`。

In [ ]:
import pandas as pd

df = pd.read_csv(RAW_DIR / "worldcities.csv")
print("資料筆數與欄位數：", df.shape)
df.head()

## 5. 繪圖測試（確認中文正常顯示）

畫出資料中城市筆數最多的 10 個國家。圖表中的中文若正常顯示，環境就設定完成了。

In [ ]:
top10 = df["country"].value_counts().head(10).sort_values()

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(top10.index, top10.values)
ax.set_title("城市筆數最多的 10 個國家")
ax.set_xlabel("城市筆數")
plt.tight_layout()
plt.show()

## 使用提醒

- **原始資料唯讀：** 不要修改 `data/raw/` 的原始資料。清理後的資料存到 `PROCESSED_DIR`，圖表存到 `OUTPUT_DIR`。
- **Agent 修改 notebook 時：** 如果 AI agent 在本機修改了這個資料夾裡的 notebook，請先關閉 Colab 分頁，等雲端硬碟同步完成後再重新開啟，避免 Colab 自動儲存把 agent 的修改蓋掉。